# CCS_RL · 船舶航迹 Demo (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZZZPhaethon/CCS_RL/blob/main/examples/colab_vessel_trajectory_demo.ipynb)

在 Colab 上跑通 **Northern Lights Phase 1** 船运 CCS 场景的物理层仿真，
并把船舶航迹可视化出来。

**这个 notebook 会做什么**

1. 浅克隆本仓库并安装唯一的外部依赖 `searoute`
2. 用**规则控制器**驱动 4 条运输船，仿真 30 天（720 小时）
3. 画出静态图：航迹地图 + 船舶作业状态时间线
4. 内嵌**可交互 Leaflet 地图**（带时间轴，可逐小时回放船位）
5. 导出完整 HTML 仪表盘供下载

**运行环境**：CPU runtime 即可，不需要 GPU。全程约 1–2 分钟，
其中仿真本身只占几秒。

> 仿真所需的场景与排放数据（`scenarios/`、`data/capture_rates/`）都已随仓库提供，
> 不需要额外下载数据集。


In [ ]:
# 1 · 环境准备
import importlib.util, os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ZZZPhaethon/CCS_RL.git"


def _in_colab() -> bool:
    # 注意：不能用 find_spec("google.colab")——非 Colab 环境没有 google 包时它会抛
    # ModuleNotFoundError 而不是返回 None。
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


IN_COLAB = _in_colab()


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if IN_COLAB:
    REPO_DIR = Path("/content/CCS_RL")
    if not REPO_DIR.exists():
        # 完整历史约 400 MB，--depth 1 只取最新快照（~90 MB），快很多
        sh("git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR))
else:
    # 本地运行：从当前目录向上找到含 src/sim 的仓库根
    here = Path.cwd().resolve()
    REPO_DIR = next(
        (p for p in (here, *here.parents) if (p / "src" / "sim").is_dir()), here
    )

# 航迹可视化这条路径只需要 searoute（离线航线图）。
# numpy / torch / gymnasium / stable-baselines3 只有 RL 训练才用得上，这里不碰。
if importlib.util.find_spec("searoute") is None:
    sh(sys.executable, "-m", "pip", "install", "-q", "searoute")

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)

import sim  # noqa: F401

print("repo :", REPO_DIR)
print("ready")

In [ ]:
# 2 · 跑仿真（规则控制器驱动 30 天）
import time
from collections import Counter

from sim.control.rule_based import RuleBasedActionGenerator
from sim.visualization import build_northern_lights_phase1_trajectory

HOURS = 24 * 30           # 30 天；改小可加快，改大可看更多航次
UNDERWAY = ("outbound_to_terminal", "return_to_origin")


def rule_based_factory(network, routes):
    return RuleBasedActionGenerator(network, routes)


t0 = time.perf_counter()
traj = build_northern_lights_phase1_trajectory(
    hours=HOURS, action_generator_factory=rule_based_factory
)
print(f"simulated {HOURS} h in {time.perf_counter() - t0:.1f} s")


def collect_tracks(traj):
    """把每帧的 vessel_positions 整理成按船分组的航迹。"""
    frames = traj["frames"]
    vessel_ids = sorted({v for f in frames for v in f["vessel_positions"]})
    tracks = {v: {"lat": [], "lon": [], "t": [], "leg": []} for v in vessel_ids}
    for f in frames:
        for vid, pos in f["vessel_positions"].items():
            tr = tracks[vid]
            tr["lat"].append(pos["lat"])
            tr["lon"].append(pos["lon"])
            tr["t"].append(f["time_h"])
            tr["leg"].append(pos["leg"])
    return vessel_ids, tracks


vessel_ids, tracks = collect_tracks(traj)

print(f"\nroute provider : {traj['route']['provider']}")
print(f"frames         : {len(traj['frames'])} (每帧 {traj['time_step_hours']:g} h)")
print(f"sites on map   : {len(traj['map']['locations'])}\n")

for vid in vessel_ids:
    legs = tracks[vid]["leg"]
    underway = sum(1 for l in legs if l in UNDERWAY)
    voyages = sum(
        1 for a, b in zip(legs, legs[1:]) if a not in UNDERWAY and b in UNDERWAY
    )
    top = Counter(legs).most_common(1)[0]
    print(f"  {vid:20s} 航行 {underway:4d} h | 航次 {voyages:2d} | 最长状态 {top[0]}")

In [ ]:
# 3 · 静态图：航迹地图 + 作业状态时间线
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

LEG_STYLE = {
    "loading_at_origin":     ("#d9a441", "Loading at emitter"),
    "outbound_to_terminal":  ("#2f6f9f", "Outbound (laden)"),
    "unloading_at_terminal": ("#4a8a5a", "Unloading at terminal"),
    "return_to_origin":      ("#b0b7bd", "Return (ballast)"),
}
VESSEL_COLORS = ["#2f6f9f", "#c1553b", "#4a8a5a", "#8a6bab", "#d9a441"]

fig = plt.figure(figsize=(15.5, 8.2))
gs = fig.add_gridspec(1, 2, wspace=0.30)
ax_map = fig.add_subplot(gs[0, 0])
ax_tl = fig.add_subplot(gs[0, 1])

# --- 左：航迹地图（只画在航段的点，停泊点不画以免糊成一团）---
for i, vid in enumerate(vessel_ids):
    tr = tracks[vid]
    idx = [j for j, lg in enumerate(tr["leg"]) if lg in UNDERWAY]
    ax_map.plot(
        [tr["lon"][j] for j in idx], [tr["lat"][j] for j in idx],
        ".", ms=2.4, alpha=0.65, color=VESSEL_COLORS[i % len(VESSEL_COLORS)],
        label=vid, zorder=3,
    )

placed = []
for loc_id, loc in traj["map"]["locations"].items():
    lat, lon = float(loc["lat"]), float(loc["lon"])
    ax_map.plot([lon], [lat], marker="s", ms=6, color="#1a1a1a", zorder=5)
    # 同一片海域的设施标签会重叠，近距离的只保留一个
    if any(abs(lat - p[0]) < 0.7 and abs(lon - p[1]) < 2.0 for p in placed):
        continue
    placed.append((lat, lon))
    label = str(loc.get("label", loc_id)).split("(")[0].strip()
    if len(label) > 22:
        label = label[:20] + "\u2026"
    ax_map.annotate(
        label, (lon, lat), textcoords="offset points", xytext=(8, 5), fontsize=8,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
        zorder=6,
    )

bbox = traj["map"]["bbox"]
ax_map.set_xlim(bbox["min_lon"], bbox["max_lon"])
ax_map.set_ylim(bbox["min_lat"], bbox["max_lat"])
ax_map.set_xlabel("Longitude (\u00b0E)")
ax_map.set_ylabel("Latitude (\u00b0N)")
ax_map.set_title(f"Vessel tracks \u2014 {HOURS} h ({HOURS // 24} days)", fontsize=11)
ax_map.grid(alpha=0.25, lw=0.6)
ax_map.legend(loc="lower right", fontsize=8, framealpha=0.92, markerscale=3.5)

# --- 右：每条船的作业状态甘特图 ---
for i, vid in enumerate(vessel_ids):
    legs, times = tracks[vid]["leg"], tracks[vid]["t"]
    start = 0
    for j in range(1, len(legs) + 1):
        if j == len(legs) or legs[j] != legs[start]:
            t0_, t1_ = times[start], (times[j] if j < len(times) else times[-1] + 1)
            ax_tl.barh(
                i, t1_ - t0_, left=t0_, height=0.6,
                color=LEG_STYLE.get(legs[start], ("#cccccc", ""))[0], edgecolor="none",
            )
            start = j

ax_tl.set_yticks(range(len(vessel_ids)))
ax_tl.set_yticklabels(vessel_ids, fontsize=9)
ax_tl.invert_yaxis()
ax_tl.set_xlim(0, HOURS)
ax_tl.set_xlabel("Simulation hour")
ax_tl.set_title("Vessel operating state over time", fontsize=11)
ax_tl.grid(axis="x", alpha=0.25, lw=0.6)
ax_tl.legend(
    handles=[Patch(facecolor=c, label=l) for c, l in LEG_STYLE.values()],
    loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=4, fontsize=8.5,
    frameon=False,
)

fig.suptitle(f"{traj['title']} \u2014 rule-based controller", fontsize=13)
fig.subplots_adjust(left=0.06, right=0.985, top=0.90, bottom=0.16)
fig.savefig("vessel_tracks.png", dpi=130)
plt.show()
print("saved -> vessel_tracks.png")

## 4 · 可交互航迹地图

下面把仓库自带的 Leaflet 仪表盘直接嵌进 notebook。拖动时间轴即可逐小时回放
船位、装卸状态与管网流量。

内嵌用的是 72 小时版本（约 1 MB），避免把 notebook 撑得过大；
30 天完整版在最后一格导出下载。

> 地图底图来自 OpenStreetMap，Leaflet 从 CDN 加载，**需要联网**。
> 如果 Colab 的输出沙箱拦了外部资源导致地图空白，静态图那一格不受影响。


In [ ]:
# 4 · 内嵌可交互仪表盘（72 h）
import html as html_mod

from IPython.display import HTML, display
from sim.visualization import render_dashboard_html

traj_short = build_northern_lights_phase1_trajectory(
    hours=72, action_generator_factory=rule_based_factory
)
doc = render_dashboard_html(traj_short)
print(f"dashboard html: {len(doc) / 1024:.0f} KB")

# 用 iframe srcdoc 包一层：既避免整份 HTML 文档污染 notebook 的 DOM，
# 也让仪表盘自带的 CSS/JS 与 Colab 页面互不干扰。
display(HTML(
    f'<iframe srcdoc="{html_mod.escape(doc)}" width="100%" height="720" '
    'style="border:1px solid #ddd;border-radius:8px"></iframe>'
))

In [ ]:
# 5 · 导出完整仪表盘（30 天）并下载
from pathlib import Path

from sim.visualization import write_northern_lights_phase1_dashboard

out = write_northern_lights_phase1_dashboard(
    Path("vessel_dashboard_30d.html"),
    hours=HOURS,
    action_generator_factory=rule_based_factory,
)
print(f"{out}  ({out.stat().st_size / 1024 / 1024:.1f} MB)")

if IN_COLAB:
    from google.colab import files

    files.download(str(out))          # 浏览器弹出下载
    files.download("vessel_tracks.png")
else:
    print("本地运行：直接用浏览器打开上面的 HTML 即可")

## 下一步

- **换场景**：把 `build_northern_lights_phase1_trajectory` 换成
  `build_northern_lights_phase2_trajectory`，可看 Phase 2 的更大船队。
- **换控制器**：`action_generator_factory` 接受任何实现了
  `next_action_frame(state)` 的对象。仓库里的 MILP、PPO、Iterative-Q
  控制器都可以接进来，用同一套可视化对比不同策略的航迹与船期。
- **不用控制器**：改传 `action_frames=[...]`，手工编排每小时动作，
  参考 [`examples/build_physical_dashboard.py`](../examples/build_physical_dashboard.py)。
